# W5: 14일 매출 예측

일별 매출에 대해 rolling mean 기반 14일 이동평균 예측선을 그립니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

dates = pd.date_range("2026-01-01", periods=30, freq="D")
np.random.seed(0)
sales = pd.DataFrame({
    "일자": dates,
    "매출": np.random.randint(80000, 170000, len(dates))
})

sales["예측_14MA"] = sales["매출"].rolling(window=14, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(8,3))
ax.plot(sales["일자"], sales["매출"], label="실제매출")
ax.plot(sales["일자"], sales["예측_14MA"], label="14일 이동평균")
ax.set_title("14일 매출 예측")
ax.legend()
plt.tight_layout()
plt.show()
